# Score centering in PyTorch

The walkthrough from [score_centering.md](score_centering.md), one state, `V = 3`. Autograd computes every `dz`,
so the hand formulas are checked against it.

In [1]:
import torch
torch.set_printoptions(precision=3, sci_mode=False)

y_hat = torch.tensor([0.5, 0.3, 0.2])     # (V,)  trainer's probabilities at this position
q     = torch.tensor([0.6, 0.3, 0.1])     # (V,)  engine's probabilities: the engine picks the token
R     = torch.tensor([1.0, 0.0, 0.0])     # (V,)  reward if token 0, 1, 2 is picked
b     = 0.25                              # ()    baseline: mean reward of this prompt's rollouts (4 rollouts, 1 correct)
A     = R - b                             # (V,)  [0.75, -0.25, -0.25]
V     = 3
onehot = torch.eye(V)                     # (V, V) row k = one-hot of token k


def dz_of(loss_fn, k):
    """Gradient of loss_fn on the logits when token k is picked -> (V,)."""
    z = torch.log(y_hat).clone().requires_grad_()   # (V,)  logits whose softmax is y_hat
    loss_fn(z, k).backward()
    return z.grad

## 1. dz for each possible pick, from autograd

Plain REINFORCE loss for sampled token `k`: `-A[k] * log softmax(z)[k]`. Its gradient on the logits is
`A[k] * (y_hat - onehot(k))`.

In [2]:
def reinforce_loss(z, k):
    """z: (V,) logits, k: sampled token -> loss ()."""
    return -A[k] * torch.log_softmax(z, -1)[k]

dz_plain = torch.stack([dz_of(reinforce_loss, k) for k in range(V)])   # (V, V) row k = dz if token k is picked
print("dz per pick (autograd):\n", dz_plain)
print("matches A[k] * (y_hat - onehot(k)):", torch.allclose(dz_plain, A.reshape(-1, 1) * (y_hat - onehot)))

dz per pick (autograd):
 tensor([[-0.375,  0.225,  0.150],
        [-0.125,  0.175, -0.050],
        [-0.125, -0.075,  0.200]])
matches A[k] * (y_hat - onehot(k)): True


## 2. Average dz at this state, drift and signal

The engine picks token `k` with probability `q[k]`, so the average update is `q @ dz_plain`.

In [3]:
average_dz = q @ dz_plain                       # (V,)  sum over k of q[k] * dz_plain[k]
A_avg      = q @ A                              # ()    advantage averaged over the engine's picks
s_bar      = q - y_hat                          # (V,)  average score onehot(k) - y_hat under the engine
drift      = -A_avg * s_bar                     # (V,)  = A_avg * (y_hat - q)
signal     = -(q * (A - A_avg))                 # (V,)
print("average dz:", average_dz)                # [-0.275, 0.18, 0.095]
print("A_avg     :", A_avg)                     # 0.35
print("drift     :", drift)                     # [-0.035, 0, 0.035]
print("signal    :", signal)                    # [-0.24, 0.18, 0.06]
print("drift + signal == average dz:", torch.allclose(drift + signal, average_dz))

average dz: tensor([-0.275,  0.180,  0.095])
A_avg     : tensor(0.350)
drift     : tensor([-0.035, -0.000,  0.035])
signal    : tensor([-0.240,  0.180,  0.060])
drift + signal == average dz: True


## 3. Score centering, from autograd

Subtract the average score `s_bar` from each pick's score. As a loss: `-A[k] * (log softmax(z)[k] - sum_v
(q - y_hat)[v] * log softmax(z)[v])`, with `q - y_hat` detached. This is miles's loss (`estimator.py:90`) with the
full vocabulary and no importance weight. Its gradient on the logits is `A[k] * (q - onehot(k))`.

In [4]:
def score_centering_loss(z, k):
    """z: (V,) logits, k: sampled token -> loss ()."""
    log_p = torch.log_softmax(z, -1)                              # (V,)
    p = log_p.exp().detach()                                      # (V,)  trainer probabilities, no gradient
    correction = ((q - p) * log_p).sum()                          # ()    sum_v (q - y_hat)[v] * log p[v]
    return -A[k] * (log_p[k] - correction)

dz_sc = torch.stack([dz_of(score_centering_loss, k) for k in range(V)])   # (V, V)
print("centered dz per pick (autograd):\n", dz_sc)
print("matches A[k] * (q - onehot(k)):", torch.allclose(dz_sc, A.reshape(-1, 1) * (q - onehot)))
print("subtracted term A[k] * (y_hat - q):\n", dz_plain - dz_sc)
print("average centered dz:", q @ dz_sc, " = signal:", torch.allclose(q @ dz_sc, signal))

centered dz per pick (autograd):
 tensor([[-0.300,  0.225,  0.075],
        [-0.150,  0.175, -0.025],
        [-0.150, -0.075,  0.225]])
matches A[k] * (q - onehot(k)): True
subtracted term A[k] * (y_hat - q):
 tensor([[-0.075,  0.000,  0.075],
        [ 0.025, -0.000, -0.025],
        [ 0.025,  0.000, -0.025]])
average centered dz: tensor([-0.240,  0.180,  0.060])  = signal: True


## 4. The baseline changes the drift, not the signal (Table 1)

In [5]:
for b_ in [0.0, 0.25, 0.6, 1.0]:
    A_ = R - b_                                                    # (V,)
    plain = q @ (A_.reshape(-1, 1) * (y_hat - onehot))             # (V,)  average plain dz
    centered = q @ (A_.reshape(-1, 1) * (q - onehot))              # (V,)  average centered dz
    A_avg_ = q @ A_                                                # ()
    print(f"b={b_:4}  A_avg={A_avg_.item():+.2f}  plain={plain}  drift={A_avg_ * (y_hat - q)}  centered={centered}")

b= 0.0  A_avg=+0.60  plain=tensor([-0.300,  0.180,  0.120])  drift=tensor([-0.060,  0.000,  0.060])  centered=tensor([-0.240,  0.180,  0.060])
b=0.25  A_avg=+0.35  plain=tensor([-0.275,  0.180,  0.095])  drift=tensor([-0.035,  0.000,  0.035])  centered=tensor([-0.240,  0.180,  0.060])
b= 0.6  A_avg=-0.00  plain=tensor([-0.240,  0.180,  0.060])  drift=tensor([ 0.000, -0.000, -0.000])  centered=tensor([-0.240,  0.180,  0.060])
b= 1.0  A_avg=-0.40  plain=tensor([-0.200,  0.180,  0.020])  drift=tensor([ 0.040, -0.000, -0.040])  centered=tensor([-0.240,  0.180,  0.060])
